# ASTM D854 — Soil Specific Gravity Lab

Colab-native interactive application using `ipywidgets`. No Streamlit server, local URL, or tunnel is required. Run the cells from top to bottom.

**Important:** generated datasets are simulated practice data. Verify calculations against the current controlled ASTM edition and laboratory SOP before formal reporting.

In [ ]:
%pip -q install ipywidgets pandas numpy matplotlib openpyxl reportlab pillow

In [ ]:
# Google Colab-native GUI for ASTM D854 soil specific gravity (water pycnometer)
# Run in Colab after installing dependencies, then call launch_app().
import io, os, math, hashlib, textwrap
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4, landscape
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
    REPORTLAB_OK = True
except Exception:
    REPORTLAB_OK = False

RAW_COLS = ['Trial', 'Dry pycnometer mass Mp (g)', 'Calibrated volume Vp (mL)', 'Dry soil mass Ms (g)', 'Pycnometer + water + soil Mrws (g)', 'Test temperature T (°C)']

CSS = '''
<style>
.sg-hero{background:linear-gradient(120deg,#102a43,#176b9b 62%,#19a7a1);padding:24px 28px;border-radius:20px;color:white;margin:8px 0 16px 0;box-shadow:0 10px 26px #102a4320}
.sg-hero h1{color:white;margin:0;font-size:30px}.sg-hero p{color:#e5f5ff;margin:8px 0 0;font-size:15px}
.sg-note{background:#f1f7fb;border-left:5px solid #176b9b;padding:12px 15px;border-radius:8px;margin:8px 0}
.sg-warn{background:#fff5e9;border-left:5px solid #f87967;padding:12px 15px;border-radius:8px;margin:8px 0}
.sg-card{border:1px solid #dfe8ef;border-radius:13px;padding:14px;margin:6px 0;background:#fff}
</style>'''


def water_density(T):
    T = np.asarray(T, dtype=float)
    return (999.842594 + 6.793952e-2*T - 9.095290e-3*T**2 + 1.001685e-4*T**3 - 1.120083e-6*T**4 + 6.536332e-9*T**5) / 1000.0


def temp_coefficient(T):
    return water_density(T) / float(water_density(20.0))


def calculate(df):
    out = df.copy()
    for c in RAW_COLS[1:]:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    T = out['Test temperature T (°C)'].to_numpy(dtype=float)
    out['Water density rho_w (g/mL)'] = water_density(T)
    out['K to 20°C'] = temp_coefficient(T)
    out['Pycnometer + water Mpw,t (g)'] = out['Dry pycnometer mass Mp (g)'] + out['Calibrated volume Vp (mL)'] * out['Water density rho_w (g/mL)']
    denom = out['Pycnometer + water Mpw,t (g)'] + out['Dry soil mass Ms (g)'] - out['Pycnometer + water + soil Mrws (g)']
    out['Gt at test temperature'] = np.where(denom > 0, out['Dry soil mass Ms (g)'] / denom, np.nan)
    out['Gs at 20°C'] = out['Gt at test temperature'] * out['K to 20°C']
    return out


def generate_data(student_id, method, n=5):
    seed = int(hashlib.sha256(str(student_id).encode()).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed)
    rows=[]
    true_gs=float(rng.normal(2.68,0.025))
    for i in range(1,n+1):
        mp=round(68.25+rng.normal(0,0.015),2)
        vp=round(250.00+rng.normal(0,0.025),2)
        ms=round((45.0 if 'Method B' in method else 47.0)+rng.normal(0,0.30),2)
        T=round(float(rng.uniform(19.0,22.0)),1)
        mpw=mp+vp*float(water_density(T))
        gt=true_gs/float(temp_coefficient(T))
        mrws=round(mpw+ms-ms/gt+rng.normal(0,0.012),2)
        rows.append([i,mp,vp,ms,mrws,T])
    return pd.DataFrame(rows,columns=RAW_COLS)


def make_excel(raw, result, meta):
    buf=io.BytesIO()
    with pd.ExcelWriter(buf,engine='openpyxl') as writer:
        pd.DataFrame([meta]).to_excel(writer,index=False,sheet_name='Student and Test Info')
        raw.to_excel(writer,index=False,sheet_name='Editable Raw Data')
        result.to_excel(writer,index=False,sheet_name='Calculated Results')
        vals=result['Gs at 20°C'].replace([np.inf,-np.inf],np.nan).dropna()
        pd.DataFrame({'Metric':['Mean Gs at 20°C','Sample standard deviation','Minimum','Maximum','Valid trials'],
                      'Value':[vals.mean() if len(vals) else np.nan, vals.std(ddof=1) if len(vals)>1 else np.nan,
                               vals.min() if len(vals) else np.nan, vals.max() if len(vals) else np.nan,len(vals)]}).to_excel(writer,index=False,sheet_name='Summary')
    return buf.getvalue()


def make_pdf(raw,result,meta,fig_bytes=None):
    if not REPORTLAB_OK: return None
    buf=io.BytesIO(); doc=SimpleDocTemplate(buf,pagesize=landscape(A4),rightMargin=10*mm,leftMargin=10*mm,topMargin=10*mm,bottomMargin=10*mm)
    styles=getSampleStyleSheet()
    styles.add(ParagraphStyle(name='TitleBlueSG',parent=styles['Title'],textColor=colors.HexColor('#102a43'),fontSize=18,leading=22))
    story=[Paragraph('ASTM D854 | Soil Specific Gravity Report',styles['TitleBlueSG']),
           Paragraph('Water pycnometer educational calculation report. Confirm the current controlled standard and laboratory SOP before formal use.',styles['Normal']),Spacer(1,5*mm)]
    info=[[str(k),str(v)] for k,v in meta.items()]
    t=Table(info,colWidths=[45*mm,205*mm]); t.setStyle(TableStyle([('BACKGROUND',(0,0),(0,-1),colors.HexColor('#eaf3fa')),('GRID',(0,0),(-1,-1),0.35,colors.HexColor('#ccd8e2')),('FONTSIZE',(0,0),(-1,-1),8),('VALIGN',(0,0),(-1,-1),'TOP')])); story += [t,Spacer(1,4*mm)]
    vals=result['Gs at 20°C'].replace([np.inf,-np.inf],np.nan).dropna()
    mean=vals.mean() if len(vals) else float('nan'); sd=vals.std(ddof=1) if len(vals)>1 else float('nan')
    story.append(Paragraph(f'<b>Mean Gs at 20°C:</b> {mean:.4f} &nbsp;&nbsp; <b>Sample SD:</b> {sd:.4f} &nbsp;&nbsp; <b>Valid trials:</b> {len(vals)}',styles['Normal']))
    cols=['Trial','Dry pycnometer mass Mp (g)','Calibrated volume Vp (mL)','Dry soil mass Ms (g)','Pycnometer + water + soil Mrws (g)','Test temperature T (°C)','Gt at test temperature','K to 20°C','Gs at 20°C']
    data=[[str(c) for c in cols]]
    for _,r in result.iterrows():
        data.append([f'{r[c]:.3f}' if isinstance(r[c],(float,np.floating)) and pd.notna(r[c]) else str(r[c]) for c in cols])
    widths=[12*mm,28*mm,25*mm,22*mm,36*mm,22*mm,25*mm,18*mm,20*mm]
    tbl=Table(data,colWidths=widths,repeatRows=1)
    tbl.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#176b9b')),('TEXTCOLOR',(0,0),(-1,0),colors.white),('GRID',(0,0),(-1,-1),0.3,colors.HexColor('#ccd8e2')),('FONTSIZE',(0,0),(-1,-1),6.2),('ALIGN',(0,0),(-1,-1),'CENTER'),('VALIGN',(0,0),(-1,-1),'MIDDLE'),('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#f2f7fb')])]))
    story += [Spacer(1,3*mm),tbl,Spacer(1,4*mm),Paragraph('Equations: Mpw,t = Mp + Vp·rho_w(T); Gt = Ms / (Mpw,t + Ms − Mrws,t); G20 = K·Gt, with K = rho_w(T)/rho_w(20°C). Polynomial water-density approximation is used. Synthetic data must not be presented as observed measurements.',styles['Normal'])]
    if fig_bytes:
        story += [PageBreak(),Paragraph('Corrected specific gravity by trial',styles['Heading2']),RLImage(io.BytesIO(fig_bytes),width=230*mm,height=115*mm)]
    doc.build(story); return buf.getvalue()


def launch_app(image_path=None):
    """Launch the Colab-native ipywidgets GUI; no Streamlit server or tunnel required."""
    state={'simulated':True,'owner':'20260001','raw':None,'last_result':None,'fig_bytes':None}
    student_id=widgets.Text(value='20260001',description='Student ID:',layout=widgets.Layout(width='320px'))
    student_name=widgets.Text(value='Student',description='Name:',layout=widgets.Layout(width='320px'))
    course=widgets.Text(value='Geotechnical Engineering Laboratory',description='Course:',layout=widgets.Layout(width='480px'))
    sample_id=widgets.Text(value='S-01',description='Sample ID:',layout=widgets.Layout(width='320px'))
    soil_type=widgets.Dropdown(options=['Sand (SP / SP-SM)','Silty or clayey sand (SM / SC)','Silt or clay (ML / CL / CH)','Other'],value='Sand (SP / SP-SM)',description='Soil:',layout=widgets.Layout(width='390px'))
    method=widgets.Dropdown(options=['Method A — moist specimen (preferred for specified soils)','Method B — oven-dry specimen'],description='Method:',layout=widgets.Layout(width='600px'))
    test_date=widgets.DatePicker(description='Test date:',value=date.today(),layout=widgets.Layout(width='320px'))
    notes=widgets.Textarea(description='Notes:',layout=widgets.Layout(width='600px',height='70px'))
    generate_btn=widgets.Button(description='Generate practice dataset',button_style='info',icon='random')
    refresh_btn=widgets.Button(description='Recalculate / refresh analysis',button_style='primary',icon='refresh')
    export_excel_btn=widgets.Button(description='Prepare Excel',button_style='success',icon='file-excel')
    export_pdf_btn=widgets.Button(description='Prepare PDF',button_style='success',icon='file-pdf')
    csv_btn=widgets.Button(description='Download raw CSV',button_style='',icon='download')
    status=widgets.Output()
    raw_out=widgets.Output()
    analysis_out=widgets.Output()
    lesson_out=widgets.Output()
    visual_out=widgets.Output()
    readme_out=widgets.Output()
    rows_box=widgets.VBox()
    quiz_widgets=[]
    questions=[
      ('Which method is preferred for specified soil types in the supplied D 854–02 guide?',['Method A: moist specimen','Method B: oven-dry specimen','Always either','Gas pycnometer only'],0,'The supplied guide identifies Method A as preferred and required for certain soil types.'),
      ('What is the No. 4 sieve opening?',['0.075 mm','2.00 mm','4.75 mm','9.5 mm'],2,'The supplied guide refers to soil passing the 4.75 mm No. 4 sieve.'),
      ('Why is deairing important?',['Increase soil mass','Remove entrapped air','Dry the specimen completely','Calibrate the balance'],1,'Entrapped air can bias the water-displacement mass balance.'),
      ('What is the reference temperature for reporting Gs?',['0°C','15°C','20°C','100°C'],2,'The result is standardized to 20°C.'),
      ('What does the student-ID generator create?',['Certified results','Simulated practice data','Calibration certificate','A substitute for testing'],1,'Generated values are synthetic and must not be represented as real observations.'),
      ('Which editable values directly affect calculated results?',['Only student name','Masses, calibrated volume and temperature','Only graph title','None'],1,'The measurements in the raw dataset are used in the mass balance and temperature correction.')]

    def metadata():
        return {'Student ID':student_id.value,'Student name':student_name.value,'Course / laboratory':course.value,'Sample ID':sample_id.value,'Soil description':soil_type.value,'Method':method.value,'Test date':str(test_date.value or date.today()),'Notes':notes.value,'Dataset status':'Simulated practice data' if state['simulated'] else 'User-entered/edited data','Reference basis':'User-supplied ASTM D 854–02 guide; verify current edition'}

    def render_raw_table(df):
        row_widgets=[]
        headers=[widgets.HTML(value=f'<b>{c}</b>',layout=widgets.Layout(width=w)) for c,w in zip(RAW_COLS,['55px','145px','135px','120px','190px','130px'])]
        row_widgets.append(widgets.HBox(headers))
        for _,r in df.iterrows():
            trial=widgets.IntText(value=int(r['Trial']),layout=widgets.Layout(width='55px'))
            mp=widgets.FloatText(value=float(r[RAW_COLS[1]]),step=0.01,layout=widgets.Layout(width='145px'))
            vp=widgets.FloatText(value=float(r[RAW_COLS[2]]),step=0.01,layout=widgets.Layout(width='135px'))
            ms=widgets.FloatText(value=float(r[RAW_COLS[3]]),step=0.01,layout=widgets.Layout(width='120px'))
            mrws=widgets.FloatText(value=float(r[RAW_COLS[4]]),step=0.01,layout=widgets.Layout(width='190px'))
            temp=widgets.FloatText(value=float(r[RAW_COLS[5]]),step=0.1,layout=widgets.Layout(width='130px'))
            row_widgets.append(widgets.HBox([trial,mp,vp,ms,mrws,temp]))
        rows_box.children=tuple(row_widgets)

    def read_widget_data():
        records=[]
        for row in rows_box.children[1:]:
            vals=[w.value for w in row.children]
            records.append(vals)
        return pd.DataFrame(records,columns=RAW_COLS)

    def draw_readme():
        with readme_out:
            clear_output(wait=True)
            display(HTML('''<div class="sg-card"><h3>Welcome to the Colab Soil Specific Gravity Lab</h3><p>This notebook-native app supports teaching and calculation for the water-pycnometer method. It runs directly inside Google Colab using ipywidgets; no Streamlit server, localhost link, or tunnel is needed.</p><h4>Tabs</h4><ul><li><b>Read Me:</b> workflow, assumptions and responsible use.</li><li><b>ASTM Visual Guide:</b> supplied image and critical procedural reminders.</li><li><b>Interactive Lesson:</b> quiz with immediate answer feedback.</li><li><b>Raw Dataset:</b> editable mass, volume and temperature values.</li><li><b>Advanced Analysis:</b> corrected results, charts, validation and exports.</li></ul><h4>Workflow</h4><ol><li>Enter student and sample information.</li><li>Generate simulated practice data or edit the cells with actual readings.</li><li>Click <b>Recalculate / refresh analysis</b>.</li><li>Review validation warnings, charts, and export files.</li></ol></div><div class="sg-warn"><b>Important:</b> generated data are synthetic practice data, not observations. This app uses a polynomial approximation for water density. Verify all equations, rounding, calibration, and the current controlled ASTM edition / laboratory SOP before formal reporting.</div>'''))

    def draw_visual():
        with visual_out:
            clear_output(wait=True)
            display(HTML('<h3>ASTM Visual Guide</h3><p>The image is a quick-reference summary only. The controlled standard and approved laboratory SOP govern test execution.</p>'))
            p=image_path
            if p and os.path.exists(p):
                from IPython.display import Image
                display(Image(filename=p, width=1050))
            else:
                display(HTML('<div class="sg-note">Image file not found. Optional: upload <code>Soil_Specific_Gravity_Testing_Guide.png</code> in the Colab Files panel and set <code>image_path</code> to its path before calling <code>launch_app(image_path=...)</code>.</div>'))
            display(HTML('''<div class="sg-card"><h4 style="color:#176b9b">Method A · Moist specimen</h4><p>The supplied guide marks this method as preferred for specified soils, including organic, highly plastic, tropical and halloysite-containing soils.</p></div><div class="sg-card"><h4 style="color:#d95f50">Method B · Oven-dry specimen</h4><p>The supplied guide describes an oven-dried specimen, with drying conditions and suitability governed by the referenced edition and lab SOP.</p></div><div class="sg-card"><h4>Critical controls</h4><ul><li>Use a suitable calibrated pycnometer and appropriate water.</li><li>Remove entrapped air with a permitted method.</li><li>Allow thermal equilibrium before final readings.</li><li>Record temperature and standardize the result to 20°C.</li><li>Check the scope and sieve-size limitations in the controlled standard.</li></ul></div>'''))
            display(HTML('<h4>Calculation equations</h4>'))
            display(HTML(r'<div class="sg-note"><b>M<sub>pw,t</sub> = M<sub>p</sub> + V<sub>p</sub>ρ<sub>w</sub>(T)</b><br><b>G<sub>t</sub> = M<sub>s</sub> / (M<sub>pw,t</sub> + M<sub>s</sub> − M<sub>rws,t</sub>)</b><br><b>G<sub>20°C</sub> = K G<sub>t</sub>, &nbsp; K = ρ<sub>w</sub>(T)/ρ<sub>w</sub>(20°C)</b></div>'))

    def draw_lesson():
        with lesson_out:
            clear_output(wait=True); quiz_widgets.clear()
            display(HTML('<h3>Interactive Lesson</h3><p>Select an answer. Feedback and score update when you click Check answers.</p>'))
            for i,(q,opts,ans,why) in enumerate(questions):
                rb=widgets.RadioButtons(options=[('— Select an answer —', None)]+[(opt,opt) for opt in opts],value=None,description=f'Q{i+1}:',layout=widgets.Layout(width='100%'))
                quiz_widgets.append(rb)
                display(widgets.HTML(f'<div class="sg-card"><b>{q}</b></div>')); display(rb)
            check=widgets.Button(description='Check answers',button_style='primary',icon='check')
            quiz_result=widgets.Output()
            def check_answers(_):
                with quiz_result:
                    clear_output(wait=True); score=0; answered=0
                    for i,(q,opts,ans,why) in enumerate(questions):
                        chosen=quiz_widgets[i].value
                        if chosen is not None:
                            answered+=1
                            if chosen==opts[ans]: score+=1; display(HTML(f'<p style="color:#147d64"><b>Q{i+1}: Correct.</b> {why}</p>'))
                            else: display(HTML(f'<p style="color:#b64035"><b>Q{i+1}: Incorrect.</b> Correct answer: {opts[ans]}. {why}</p>'))
                    display(HTML(f'<h4>Score: {score} / {answered} answered</h4>'))
            check.on_click(check_answers); display(check,quiz_result)

    def validate_and_draw(_=None):
        with status:
            clear_output(wait=True)
            try:
                raw=read_widget_data()
                issues=[]
                if raw.empty: issues.append('No trial rows are present.')
                for c in RAW_COLS[1:]: raw[c]=pd.to_numeric(raw[c],errors='coerce')
                if raw[RAW_COLS[1:]].isna().any().any(): issues.append('Some required numeric cells are blank or invalid.')
                if (raw['Dry soil mass Ms (g)']<=0).any(): issues.append('Dry soil mass must be greater than zero.')
                if (raw['Calibrated volume Vp (mL)']<=0).any(): issues.append('Calibrated volume must be greater than zero.')
                if ((raw['Test temperature T (°C)']<0)|(raw['Test temperature T (°C)']>40)).any(): issues.append('Temperature outside the supported water-density approximation range of 0–40°C.')
                result=calculate(raw)
                if result['Gs at 20°C'].isna().any(): issues.append('A row has a non-positive mass-balance denominator.')
                vals=result['Gs at 20°C'].replace([np.inf,-np.inf],np.nan).dropna()
                if ((vals<1)|(vals>4)).any(): issues.append('A calculated Gs is outside broad screening range 1–4; check inputs. This is not an ASTM acceptance criterion.')
                if issues:
                    for issue in issues: display(HTML(f'<div class="sg-warn">⚠️ {issue}</div>'))
                else: display(HTML('<div class="sg-note">✓ Basic input checks passed. This does not certify laboratory compliance.</div>'))
                state['raw']=raw; state['last_result']=result
                with analysis_out:
                    clear_output(wait=True)
                    valid=result['Gs at 20°C'].replace([np.inf,-np.inf],np.nan).dropna()
                    display(HTML('<h3>Advanced Analysis</h3>'))
                    if not len(valid):
                        display(HTML('<div class="sg-warn">No valid results. Check raw data and mass balance.</div>')); return
                    mean=float(valid.mean()); sd=float(valid.std(ddof=1)) if len(valid)>1 else np.nan
                    cards=widgets.HBox([widgets.HTML(f'<div class="sg-card"><b>Mean Gs at 20°C</b><h2>{mean:.4f}</h2></div>'),widgets.HTML(f'<div class="sg-card"><b>Sample SD</b><h2>{sd:.4f}</h2></div>'),widgets.HTML(f'<div class="sg-card"><b>Valid trials</b><h2>{len(valid)}</h2></div>'),widgets.HTML(f'<div class="sg-card"><b>Range</b><h2>{valid.min():.3f}–{valid.max():.3f}</h2></div>')])
                    display(cards)
                    display(HTML('<h4>Calculated results</h4>'))
                    display(result.round(5).style.format(precision=5).set_table_styles([{'selector':'th','props':[('background','#176b9b'),('color','white'),('text-align','left')]},{'selector':'td','props':[('padding','6px')]}]))
                    fig,ax=plt.subplots(figsize=(9,4.5)); ax.plot(result['Trial'],result['Gs at 20°C'],marker='o',linewidth=2); ax.axhline(mean,linestyle='--',label=f'Mean = {mean:.4f}'); ax.set_xlabel('Trial'); ax.set_ylabel('Gs at 20°C'); ax.set_title('Corrected specific gravity by trial'); ax.grid(True,alpha=.25); ax.legend(); fig.tight_layout()
                    fbuf=io.BytesIO(); fig.savefig(fbuf,format='png',dpi=160,bbox_inches='tight'); state['fig_bytes']=fbuf.getvalue(); display(fig); plt.close(fig)
                    fig2,ax2=plt.subplots(figsize=(9,4)); ax2.scatter(result['Test temperature T (°C)'],result['Gs at 20°C'],s=65); ax2.set_xlabel('Test temperature (°C)'); ax2.set_ylabel('Gs at 20°C'); ax2.set_title('Corrected result versus test temperature'); ax2.grid(True,alpha=.25); fig2.tight_layout(); display(fig2); plt.close(fig2)
                    if state['simulated']: display(HTML('<div class="sg-warn">These are simulated practice data unless you have replaced them with actual measured readings.</div>'))
                    display(HTML('<p>Descriptive statistics are reported; the app does not automatically declare ASTM compliance based on the results.</p>'))
            except Exception as e:
                display(HTML(f'<div class="sg-warn">Could not calculate: {type(e).__name__}: {e}</div>'))

    def generate(_):
        df=generate_data(student_id.value,method.value)
        state['raw']=df; state['owner']=student_id.value; state['simulated']=True
        render_raw_table(df)
        validate_and_draw()
        with status: display(HTML(f'<div class="sg-note">Practice data generated reproducibly from Student ID {student_id.value}. These are simulated values.</div>'))

    def download_bytes(filename,data):
        try:
            from google.colab import files
            path='/content/'+filename
            with open(path,'wb') as f: f.write(data)
            files.download(path)
        except Exception as e:
            with status: display(HTML(f'<div class="sg-warn">Download helper unavailable: {e}. File saved if possible at /content/{filename}.</div>'))

    def do_excel(_):
        validate_and_draw()
        if state['last_result'] is not None:
            data=make_excel(state['raw'],state['last_result'],metadata())
            download_bytes(f'ASTM_D854_report_{student_id.value}.xlsx',data)
    def do_pdf(_):
        validate_and_draw()
        if not REPORTLAB_OK:
            with status: display(HTML('<div class="sg-warn">PDF requires reportlab. Run %pip install reportlab and try again.</div>')); return
        if state['last_result'] is not None:
            data=make_pdf(state['raw'],state['last_result'],metadata(),state['fig_bytes'])
            download_bytes(f'ASTM_D854_report_{student_id.value}.pdf',data)
    def do_csv(_):
        validate_and_draw()
        if state['raw'] is not None: download_bytes(f'ASTM_D854_raw_{student_id.value}.csv',state['raw'].to_csv(index=False).encode('utf-8'))

    generate_btn.on_click(generate); refresh_btn.on_click(validate_and_draw); export_excel_btn.on_click(do_excel); export_pdf_btn.on_click(do_pdf); csv_btn.on_click(do_csv)
    state['raw']=generate_data(student_id.value,method.value)
    render_raw_table(state['raw'])

    header=HTML(CSS+'<div class="sg-hero"><h1>🧪 Soil Specific Gravity Lab</h1><p>ASTM D854 · Water Pycnometer · Colab-native interactive learning, editable measurements, calculations and reports</p></div>')
    controls=widgets.VBox([widgets.HBox([student_id,student_name]),widgets.HBox([course,sample_id]),widgets.HBox([soil_type,method]),widgets.HBox([test_date]),notes,widgets.HBox([generate_btn,refresh_btn])])
    metadata_panel=widgets.Accordion(children=[controls]); metadata_panel.set_title(0,'Student / sample information and dataset controls'); metadata_panel.selected_index=0
    tabs=widgets.Tab(children=[readme_out,visual_out,lesson_out,widgets.VBox([widgets.HTML('<h3>Raw Dataset</h3><p>Edit the numeric fields directly, then click Recalculate / refresh analysis. All masses are grams, volume is mL, and temperature is °C.</p>'),rows_box,widgets.HBox([refresh_btn,generate_btn,csv_btn]),status]),analysis_out])
    for i,title in enumerate(['Read Me','ASTM Visual Guide','Interactive Lesson','Raw Dataset','Advanced Analysis']): tabs.set_title(i,title)
    display(header,metadata_panel,tabs,widgets.HBox([export_excel_btn,export_pdf_btn]))
    draw_readme(); draw_visual(); draw_lesson(); validate_and_draw()
    return {'tabs':tabs,'state':state,'refresh':validate_and_draw,'metadata':metadata}

print('Loaded soil_specific_gravity_colab.py. Run launch_app(image_path="/content/Soil_Specific_Gravity_Testing_Guide.png") to display the GUI.')


In [ ]:
# The user-provided visual guide is embedded in this notebook.
import base64
_image_b64 = '...'
image_path = '/content/Soil_Specific_Gravity_Testing_Guide.jpg'
with open(image_path, 'wb') as _f:
    _f.write(base64.b64decode(_image_b64))
print('Visual guide ready:', image_path)


In [ ]:
# Launch the complete interactive GUI directly below this cell.
app = launch_app(image_path=image_path)